# Exact match or string matching measure (using fuzzy logic)

In [8]:
from google.colab import drive;
drive.mount('/content/drive');
!pip install pandas


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install -U scikit-learn


In [10]:
!pip install fuzzywuzzy pandas openpyxl

In [11]:
import pandas as pd

file_path = '/content/drive/MyDrive/Colab Notebooks/IST/Example/GPT_4_Turbo/IM_Software/Result/GPT_4_Turbo_Result.xlsx'
df = pd.read_excel(file_path)
print(df.head())

     Run                                            ACAS XU  \
0  Run 1  G0: ACAS Xu satisfies security requirements\nG...   
1  Run 2  G0: ACAS Xu satisfies security requirements\nG...   
2  Run 3  G0: ACAS Xu satisfies security requirements\nG...   
3  Run 4  G0: ACAS Xu satisfies security requirements\nG...   
4  Run 5  G0: ACAS Xu satisfies security requirements\nG...   

                                            BlueROV2  \
0  G1: Hazards for BlueROV2 system addressed in a...   
1  G1: Hazards for BlueROV2 system addressed in a...   
2  G1: Hazards for BlueROV2 system addressed in a...   
3  G1: Hazards for BlueROV2 System addressed in a...   
4  G1: Hazards for BlueROV2 system addressed in a...   

                                            DeepMind  \
0  G1: Interpretability Claim. The DeepMind's ML ...   
1  G1: Interpretability Claim. The DeepMind's ML ...   
2  G1: Interpretability Claim. The DeepMind syste...   
3  G1: Interpretability Claim. The DeepMind ML mo...   
4  G

In [12]:
file_path2 = '/content/drive/MyDrive/Colab Notebooks/IST/Example/GPT_4_Turbo/IM_Software/Result/Ground_Truth_AC.xlsx'
ground_truth_df = pd.read_excel(file_path2)
print(ground_truth_df.head())

       Case                                                 AC
0   ACAS XU  G0: ACAS Xu satisfies security requirements\nG...
1  BlueROV2  G1: Hazards for BlueROV2 addressed in accordan...
2  DeepMind  G1: Interpretability Claim - The ML system is ...
3      GPCA  G1: Operational safety is verified in "GPCA sy...


In [13]:
from fuzzywuzzy import fuzz


for index, experiment_row in df.iterrows():
    for case in ['ACAS XU', 'BlueROV2', 'DeepMind', 'GPCA']:
        experiment_text = experiment_row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]
        similarity_score = fuzz.ratio(experiment_text, ground_truth_text)/ 100.0

        print(f"Similarity score for {case} in {experiment_row['Run']}: {similarity_score}")
    print("-------")

Similarity score for ACAS XU in Run 1: 0.57
Similarity score for BlueROV2 in Run 1: 0.71
Similarity score for DeepMind in Run 1: 0.23
Similarity score for GPCA in Run 1: 0.44
-------
Similarity score for ACAS XU in Run 2: 0.5
Similarity score for BlueROV2 in Run 2: 0.41
Similarity score for DeepMind in Run 2: 0.24
Similarity score for GPCA in Run 2: 0.2
-------
Similarity score for ACAS XU in Run 3: 0.49
Similarity score for BlueROV2 in Run 3: 0.76
Similarity score for DeepMind in Run 3: 0.23
Similarity score for GPCA in Run 3: 0.33
-------
Similarity score for ACAS XU in Run 4: 0.41
Similarity score for BlueROV2 in Run 4: 0.75
Similarity score for DeepMind in Run 4: 0.27
Similarity score for GPCA in Run 4: 0.24
-------
Similarity score for ACAS XU in Run 5: 0.52
Similarity score for BlueROV2 in Run 5: 0.84
Similarity score for DeepMind in Run 5: 0.22
Similarity score for GPCA in Run 5: 0.24
-------


# Bleu score

In [14]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 4.4 MB/s eta 0:00:00


In [15]:
import sacrebleu

def calculate_bleu_score(experiment_text, ground_truth_text):
    references = [[ground_truth_text]]
    system_output = [experiment_text]
    bleu = sacrebleu.corpus_bleu(system_output, references)
    # Scale the score to a range between 0 and 1 and round to two decimal points
    return round(bleu.score / 100, 2)

for case in ['ACAS XU', 'BlueROV2', 'DeepMind', 'GPCA']:
    print(f"Calculating BLEU scores for {case}:")
    for index, row in df.iterrows():
        experiment_text = row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

        if pd.notnull(experiment_text) and pd.notnull(ground_truth_text):
            bleu_score = calculate_bleu_score(experiment_text, ground_truth_text)
            print(f"Run {index + 1} BLEU score: {bleu_score}")
        else:
            print(f"Run {index + 1} BLEU score: Data Missing")
    print("-------")


Calculating BLEU scores for ACAS XU:
Run 1 BLEU score: 0.53
Run 2 BLEU score: 0.62
Run 3 BLEU score: 0.59
Run 4 BLEU score: 0.42
Run 5 BLEU score: 0.58
-------
Calculating BLEU scores for BlueROV2:
Run 1 BLEU score: 0.58
Run 2 BLEU score: 0.37
Run 3 BLEU score: 0.59
Run 4 BLEU score: 0.52
Run 5 BLEU score: 0.76
-------
Calculating BLEU scores for DeepMind:
Run 1 BLEU score: 0.23
Run 2 BLEU score: 0.21
Run 3 BLEU score: 0.22
Run 4 BLEU score: 0.25
Run 5 BLEU score: 0.2
-------
Calculating BLEU scores for GPCA:
Run 1 BLEU score: 0.3
Run 2 BLEU score: 0.23
Run 3 BLEU score: 0.35
Run 4 BLEU score: 0.3
Run 5 BLEU score: 0.31
-------


# Semantic Similarity

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

for case in ['ACAS XU', 'BlueROV2', 'DeepMind', 'GPCA']:
    experiment_texts = df[case].tolist()
    ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

    combined_texts = experiment_texts + [ground_truth_text]

    tfidf_matrix = vectorizer.fit_transform(combined_texts)

    experiment_vectors = tfidf_matrix[:-1]
    ground_truth_vector = tfidf_matrix[-1:]

    cosine_similarities = cosine_similarity(experiment_vectors, ground_truth_vector)

    for i, similarity_score in enumerate(cosine_similarities.flatten(), 1):
        # Round the score to two decimal places without multiplying by 100
        rounded_score = round(similarity_score, 2)
        print(f"Cosine similarity for {case}, Run {i}: {rounded_score}")
    print("-------")


Cosine similarity for ACAS XU, Run 1: 0.88
Cosine similarity for ACAS XU, Run 2: 0.86
Cosine similarity for ACAS XU, Run 3: 0.9
Cosine similarity for ACAS XU, Run 4: 0.82
Cosine similarity for ACAS XU, Run 5: 0.83
-------
Cosine similarity for BlueROV2, Run 1: 0.85
Cosine similarity for BlueROV2, Run 2: 0.83
Cosine similarity for BlueROV2, Run 3: 0.9
Cosine similarity for BlueROV2, Run 4: 0.81
Cosine similarity for BlueROV2, Run 5: 0.9
-------
Cosine similarity for DeepMind, Run 1: 0.54
Cosine similarity for DeepMind, Run 2: 0.54
Cosine similarity for DeepMind, Run 3: 0.59
Cosine similarity for DeepMind, Run 4: 0.64
Cosine similarity for DeepMind, Run 5: 0.63
-------
Cosine similarity for GPCA, Run 1: 0.6
Cosine similarity for GPCA, Run 2: 0.61
Cosine similarity for GPCA, Run 3: 0.54
Cosine similarity for GPCA, Run 4: 0.56
Cosine similarity for GPCA, Run 5: 0.59
-------
